# K-pop 뮤직비디오 데이터 전처리

2024년 크롤링 원본과 2023·2025년 추가 크롤링 데이터를 정리해, 중복 없는 최종 데이터셋을 만드는 노트북입니다.

**처리 순서**
- **0. 데이터 전처리** — 2024 원본 CSV의 불필요한 컬럼 정리, 장르 라벨 통일
- **1. 중복 처리 (2024 데이터)** — songName+artists 기준 중복 후보 추출 → video_id 기준 1차 제거 → 유통사 채널 제거로 곡당 1개 영상만 남김
- **2. 2023, 2025 데이터 추가** — 2024 결과에 없는 곡만 골라 결측치 정리·채널 필터링 후 추가
- **3. 최종 병합 및 저장** — 2024 결과 + 2023,2025 추가분을 합쳐 최종 CSV로 저장

**입력 파일**
- `2024최종뮤비자료(1월30일) (1).csv` — 2024년 크롤링 + 오디오/비주얼 피처 원본
- `df_combined(2023,2025파일) (1).csv` — 2023,2025년 크롤링 + genie 장르/가사 보강 데이터

**출력 파일**
- `2024뮤비자료_최종전처리완료(1월31일).csv` — 0단계 결과
- `data(drop_duplicated).csv` — 1단계 결과
- `add_potential.csv` — 2단계 중간 결과
- `전체뮤비_최종전처리완료.csv` — 3단계 결과 (파이프라인 최종 완성본)

> **참고:** Colab 환경(`/content/...` 경로) 기준으로 작성돼 로컬에서 실행하려면 경로 수정이 필요합니다. 1번 섹션 상단 안내처럼, 일부 셀 출력은 예전 파이프라인(`final_merged_cleaned_v2`) 기준 값이라 정확한 수치를 보려면 Colab에서 처음부터 재실행해야 합니다.

# 0. 데이터 전처리
<small>원본 크롤링 결과 CSV를 불러와 불필요한 인덱스 컬럼(`Unnamed: 0` 등)을 제거하고, 장르 라벨을 통일(`POP / 락` → `락`)한 뒤 전처리 완료 파일로 저장합니다.</small>

In [ ]:
import pandas as pd
df = pd.read_csv('/content/2024최종뮤비자료(1월30일) (1).csv')

In [ ]:
df.head()

,Unnamed: 0.1,Unnamed: 0,url,songName,artists,publishTime,video_id,api_view_count,api_like_count,api_comment_count,...,avg_tokens_per_line,line_count,punctuation_ratio,compression_rate,pronoun_frequency,positive_emotion_ratio,negative_emotion_ratio,korean_ratio,english_ratio,genie_genre
0,0,0,https://www.youtube.com/watch?v=ekr2nIex040,APT. (ROSÉ & Bruno Mars),684|로제 (ROSÉ)|ROSE|1,2024.10.18,ekr2nIex040,2217209894,17370428,824773,...,6.94,80.0,0.0975,0.2556,0.0883,0.0440,0.0137,0.1120,0.5032,락
1,1,1,https://www.youtube.com/watch?v=Zp-Jhuhq0bQ,DRIP,2543|BABYMONSTER (베이비몬스터)|BABYMONSTER|1,2024.11.01,Zp-Jhuhq0bQ,328137312,2995930,127426,...,5.53,110.0,0.1036,0.3965,0.0739,0.0356,0.0071,0.0893,0.4979,댄스
2,2,2,https://www.youtube.com/watch?v=Sz_wWzgh-vQ,Strategy (feat. Megan Thee Stallion),5|TWICE (트와이스)|TWICE|1,2024.12.06,Sz_wWzgh-vQ,127492278,1858130,200828,...,6.89,94.0,0.0225,0.3250,0.1653,0.0690,0.0293,0.0000,0.8017,댄스
3,3,3,https://www.youtube.com/watch?v=eA0lHNZ1KCA,toxic till the end,684|로제 (ROSÉ)|ROSE|1,2024.12.06,eA0lHNZ1KCA,122194728,2584605,112909,...,5.16,70.0,0.0071,0.3298,0.1995,0.0325,0.0214,0.0000,0.8596,락
4,4,4,https://www.youtube.com/watch?v=1kXLsrun51s,Love In My Heart,2543|BABYMONSTER (베이비몬스터)|BABYMONSTER|1,2024.12.16,1kXLsrun51s,77443006,1112609,54988,...,7.66,65.0,0.0570,0.3665,0.1246,0.0920,0.0098,0.2206,0.5089,댄스


In [ ]:
df.tail()

,Unnamed: 0.1,Unnamed: 0,url,songName,artists,publishTime,video_id,api_view_count,api_like_count,api_comment_count,...,avg_tokens_per_line,line_count,punctuation_ratio,compression_rate,pronoun_frequency,positive_emotion_ratio,negative_emotion_ratio,korean_ratio,english_ratio,genie_genre
951,959,1052,https://www.youtube.com/watch?v=m6ZC6Ua-CuU,여름아 부탁해,1462|순순희||0,2023.07.22,m6ZC6Ua-CuU,6485,123,9,...,6.75,426.0,0.0171,0.5122,0.0900,0.0332,0.0309,0.5627,0.2627,인디
952,960,1053,https://www.youtube.com/watch?v=EaI2F-ZurXI,Beautiful Stranger,2530|신유미||0,2023.10.21,EaI2F-ZurXI,5114,139,22,...,4.08,25.0,0.0130,0.5293,0.0714,0.0619,0.0476,0.5238,0.2460,락
953,961,1054,https://www.youtube.com/watch?v=GdpKIl19dPc,우린 다른 길을 걷고 있었나 봐,1391|이우||0,2023.12.21,GdpKIl19dPc,4224,127,11,...,8.38,494.0,0.0298,0.5321,0.0863,0.0431,0.0382,0.5656,0.2479,발라드
954,962,1055,https://www.youtube.com/watch?v=UhtqWv1iLqY,I의 짝사랑 (유앤미앤미 X 정효빈),1392|정효빈||0,2023.10.18,UhtqWv1iLqY,4253,108,2,...,9.25,504.0,0.0409,0.5264,0.0695,0.0345,0.0380,0.5494,0.2356,발라드
955,963,1056,https://www.youtube.com/watch?v=sAhxgrvlxk4,울었어 (Feat. 정승환),1689|딘딘|DinDin|1,2023.10.19,sAhxgrvlxk4,2563,222,22,...,5.08,66.0,0.0058,0.4451,0.1075,0.0362,0.0663,0.8225,0.0025,랩/힙합


In [ ]:
df.drop(columns = ['Unnamed: 0.1', 'Unnamed: 0'], inplace =True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 956 entries, 0 to 955
Data columns (total 58 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   url                     956 non-null    object 
 1   songName                956 non-null    object 
 2   artists                 956 non-null    object 
 3   publishTime             956 non-null    object 
 4   video_id                956 non-null    object 
 5   api_view_count          956 non-null    int64  
 6   api_like_count          956 non-null    int64  
 7   api_comment_count       956 non-null    int64  
 8   api_published_at        956 non-null    object 
 9   api_duration            956 non-null    object 
 10  api_title               956 non-null    object 
 11  api_channel_title       956 non-null    object 
 12  valence_raw             956 non-null    float64
 13  arousal_raw             956 non-null    float64
 14  valence_normalized      956 non-null    fl

In [ ]:
df['genie_genre'].unique()

array(['락', '댄스', '랩/힙합', '발라드', 'POP / 팝', '일렉트로니카', 'R&B/Soul', '인디'],
      dtype=object)

In [ ]:
df['genie_genre'] = df['genie_genre'].replace({
    'POP / 락' : '락'
})

In [ ]:
df.to_csv('2024뮤비자료_최종전처리완료(1월31일).csv', index = False)

# 1. 중복 처리 (2024 데이터)
<small>songName+artists 기준으로 중복 후보를 추출하고, video_id 기준 1차 제거 후 유통사 채널(뮤직 퍼블리셔 공식 업로드 등)을 제거해 곡당 1개 영상만 남깁니다.</small>

> **주의:** 아래 셀들의 출력(중복 개수, 행 수 등)은 원래 `final_merged_cleaned_v2`(2023~2025 통합 + 아티스트 정보까지 병합된 2545행 데이터) 기준으로 실행됐던 결과입니다. 입력을 `00_전처리` 결과물(2024년 원본, 956행)로 바꾼 뒤에는 재실행하지 않아 숫자가 실제 값과 다릅니다. 로직 자체는 songName/artists/video_id/api_channel_title/url만 사용해 그대로 유효하지만, 정확한 수치는 Colab에서 처음부터 다시 실행해야 확인할 수 있습니다.

In [1]:
import pandas as pd
final_df = pd.read_csv('2024뮤비자료_최종전처리완료(1월31일).csv')  # final_merged_cleaned_v2 대신 00_preprocessing 결과를 그대로 이어받음

In [2]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2545 entries, 0 to 2544
Data columns (total 59 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   url                     2545 non-null   object 
 1   songName                2545 non-null   object 
 2   artists                 2545 non-null   object 
 3   video_id                2545 non-null   object 
 4   api_view_count          2545 non-null   float64
 5   api_like_count          2545 non-null   float64
 6   api_comment_count       2545 non-null   float64
 7   api_published_at        2545 non-null   object 
 8   api_duration            2545 non-null   object 
 9   api_channel_title       2545 non-null   object 
 10  valence_raw             2545 non-null   float64
 11  arousal_raw             2545 non-null   float64
 12  valence_normalized      2545 non-null   float64
 13  arousal_normalized      2545 non-null   float64
 14  energy                  2545 non-null   

## 1. songName+artists 기준 중복 후보 추출
<small>곡명+아티스트가 같은 행을 모두 찾아 `df_dup`(중복 후보)와 `df_unique`(애초에 중복 없는 곡)로 나눕니다.</small>

In [3]:
# 1. 중복된 것들만 추출 (모든 행 포함)
dup_mask = final_df.duplicated(subset=['songName', 'artists'], keep=False)

In [6]:
dup_mask.sum()

np.int64(912)

In [7]:
df_dup = final_df[dup_mask].reset_index(drop=True)      # 중복 처리 필요한 것들
df_unique = final_df[~dup_mask].reset_index(drop=True)  # 처음부터 중복 없는 것들

print(f"원본: {len(final_df)} rows")
print(f"중복 없는 것 (df_unique): {len(df_unique)} rows")
print(f"중복 있는 것 (df_dup): {len(df_dup)} rows")
print(f"합계 확인: {len(df_unique) + len(df_dup)} rows")

원본: 2545 rows
중복 없는 것 (df_unique): 1633 rows
중복 있는 것 (df_dup): 912 rows
합계 확인: 2545 rows


## 2. video_id 기준 1차 중복 제거
<small>중복 후보 중 video_id가 같은 행(완전히 동일한 영상)을 먼저 하나만 남깁니다.</small>

In [8]:
df_dup_deduped = (
    df_dup
    .drop_duplicates(subset=['video_id'], keep='first')
    .reset_index(drop=True)
)

print(f"중복 제거 전: {len(df_dup)} rows")
print(f"중복 제거 후: {len(df_dup_deduped)} rows")
print(f"제거된 행: {len(df_dup) - len(df_dup_deduped)} rows")

중복 제거 전: 912 rows
중복 제거 후: 664 rows
제거된 행: 248 rows


In [12]:
df_dup_deduped.columns

Index(['url', 'songName', 'artists', 'video_id', 'api_view_count',
       'api_like_count', 'api_comment_count', 'api_published_at',
       'api_duration', 'api_channel_title', 'valence_raw', 'arousal_raw',
       'valence_normalized', 'arousal_normalized', 'energy', 'loudness',
       'tempo', 'duration_ms', 'speechiness', 'acoustic_score',
       'sentiment_compound', 'sentiment_positive', 'sentiment_negative',
       'sentiment_neutral', 'avg_brightness', 'avg_motion', 'avg_r_value',
       'avg_g_value', 'avg_b_value', 'frames_analyzed', 'genie_lyrics',
       'genie_genre', 'token_count', 'unique_token_ratio',
       'repeated_token_ratio', 'avg_token_length', 'hapax_legomenon',
       'hapax_dislegomenon', 'hapax_trislegomenon', 'unique_tokens_per_line',
       'avg_tokens_per_line', 'line_count', 'punctuation_ratio',
       'compression_rate', 'pronoun_frequency', 'korean_ratio',
       'english_ratio', 'year', 'month', 'api_duration_sec', 'lyrics_clean',
       'song_title_clea

In [14]:
df_dup_deduped['video_id'].nunique()

664

In [15]:
len(df_dup_deduped)

664

## 3. 유통사 채널 중복 확인
<small>같은 곡이 몇 개의 서로 다른 채널(`api_channel_title`)에 걸쳐 남아있는지 집계합니다.</small>

In [20]:
result = df_dup_deduped.groupby(['songName', 'artists'])['api_channel_title'].nunique()
len(result[result >= 2])

237

In [22]:
# 237개 실제로 어떤 채널들이 있는지 확인
multi_channel_songs = result[result >= 2].index
df_dup_deduped[df_dup_deduped.set_index(['songName', 'artists']).index.isin(multi_channel_songs)]\
    .groupby(['songName', 'artists'])['api_channel_title'].apply(list)

,,api_channel_title
songName,artists,
2 Months,3612|유아유 (UAU)|UAU|0,"[1theK (원더케이), Dreamcatcher official]"
ATE THAT,2756|YOUNG POSSE (영파씨)|YOUNGPOSSE|1,"[YOUNG POSSE • 영파씨, 1theK (원더케이)]"
Ain't Nobody,2932|VVUP(비비업)|VVUP|1,"[VVUP, GENIE MUSIC]"
BANG OUT,2772|WHIB(휘브)|WHIB|0,"[WHIB, 1theK (원더케이)]"
BEBE,2031|STAYC(스테이씨)|STAYC|1,"[1theK (원더케이), STAYC]"
...,...,...
행운을 부탁해,647|보라미유||0,"[쇼파르엔터테인먼트, SUPER SOUND Bugs!]"
혀끝(Stuck),2752|82MAJOR(에이티투메이저)|82MAJOR|1,"[82MAJOR, Stone Music Entertainment]"
호감,1125|윤종신|YoonJongShin|1,"[Dreamus Music, 월간 윤종신]"


In [23]:
# 어떤 채널 조합이 많이 나오는지 확인
multi_channel_songs = result[result >= 2].index
df_multi = df_dup_deduped[
    df_dup_deduped.set_index(['songName', 'artists']).index.isin(multi_channel_songs)
]

# 중복 채널로 많이 등장하는 채널명 순위
df_multi['api_channel_title'].value_counts()

,count
api_channel_title,
1theK (원더케이),138
Stone Music Entertainment,23
Dreamus Music,17
워너뮤직코리아 (Warner Music Korea),14
GENIE MUSIC,12
...,...
BAE173 [OFFICIAL],1
GUCKKASTEN Official,1
정세운 JEONGSEWOON,1


## 4. 유통사 채널 제거 (REMOVE_LIST 1차 적용)
<small>1theK, Stone Music Entertainment, Dreamus Music 등 알려진 유통사 공식 업로드 채널을 제거해 곡당 채널 수를 줄입니다.</small>

In [24]:
REMOVE_LIST = ['1theK (원더케이)', 'Stone Music Entertainment', 'Dreamus Music',
               '워너뮤직코리아 (Warner Music Korea)', 'GENIE MUSIC']

# 각 곡별로 유통사 채널이 몇 개인지 확인
def count_dist_channels(group):
    return group['api_channel_title'].isin(REMOVE_LIST).sum()

dist_count = df_dup_deduped.groupby(['songName', 'artists']).apply(count_dist_channels)
print("유통사 채널 2개 이상인 곡:")
print(dist_count[dist_count >= 2])

유통사 채널 2개 이상인 곡:
songName           artists   
교회오빠 (Feat.BOBBY)  1564|오반||0    2
dtype: int64


/tmp/ipykernel_3192/250054025.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dist_count = df_dup_deduped.groupby(['songName', 'artists']).apply(count_dist_channels)


In [25]:
df_dup_deduped[df_dup_deduped['songName'] == '교회오빠 (Feat.BOBBY)']

,url,songName,artists,video_id,api_view_count,api_like_count,api_comment_count,api_published_at,api_duration,api_channel_title,...,api_duration_sec,lyrics_clean,song_title_clean,lyrics_topic,song_title_topic,kor_name,activity_type,member_count,gender,is_group
612,https://www.youtube.com/watch?v=my-Mrod6cJU,교회오빠 (Feat.BOBBY),1564|오반||0,my-Mrod6cJU,2436.0,0.0,5.0,2025-03-05 09:00:57+00:00,PT15S,워너뮤직코리아 (Warner Music Korea),...,15,너의 교회 오빠 어때 주일에 교회도 안 갈게 저 이태원 오빠는 안되지 내가 되어줄게...,교회오빠 Feat BOBBY,2,5,오반,"남성, 솔로",1.0,남성,0.0
621,https://www.youtube.com/watch?v=oHyzyV1qOVY,교회오빠 (Feat.BOBBY),1564|오반||0,oHyzyV1qOVY,55464.0,1317.0,135.0,2025-03-13 09:00:58+00:00,PT3M27S,1theK (원더케이),...,207,너의 교회 오빠 어때 주일에 교회도 안 갈게 저 이태원 오빠는 안되지 내가 되어줄게...,교회오빠 Feat BOBBY,2,5,오반,"남성, 솔로",1.0,남성,0.0


In [26]:
df_dup_deduped[df_dup_deduped['url'] == 'https://www.youtube.com/watch?v=my-Mrod6cJU']

,url,songName,artists,video_id,api_view_count,api_like_count,api_comment_count,api_published_at,api_duration,api_channel_title,...,api_duration_sec,lyrics_clean,song_title_clean,lyrics_topic,song_title_topic,kor_name,activity_type,member_count,gender,is_group
612,https://www.youtube.com/watch?v=my-Mrod6cJU,교회오빠 (Feat.BOBBY),1564|오반||0,my-Mrod6cJU,2436.0,0.0,5.0,2025-03-05 09:00:57+00:00,PT15S,워너뮤직코리아 (Warner Music Korea),...,15,너의 교회 오빠 어때 주일에 교회도 안 갈게 저 이태원 오빠는 안되지 내가 되어줄게...,교회오빠 Feat BOBBY,2,5,오반,"남성, 솔로",1.0,남성,0.0


In [27]:
len(df_dup_deduped)

664

In [29]:
# url 기준으로 해당 행의 index 찾아서 drop
idx = df_dup_deduped[df_dup_deduped['url'] == 'https://www.youtube.com/watch?v=my-Mrod6cJU'].index
print(idx)

df_dup_deduped = df_dup_deduped.drop(idx).reset_index(drop=True)
print(f"제거 후: {len(df_dup_deduped)} rows")

Index([612], dtype='int64')
제거 후: 663 rows


In [30]:
len(df_dup_deduped)

663

In [31]:
df_dup_deduped[df_dup_deduped['url'] == 'https://www.youtube.com/watch?v=my-Mrod6cJU']

,url,songName,artists,video_id,api_view_count,api_like_count,api_comment_count,api_published_at,api_duration,api_channel_title,...,api_duration_sec,lyrics_clean,song_title_clean,lyrics_topic,song_title_topic,kor_name,activity_type,member_count,gender,is_group


In [32]:
REMOVE_LIST = ['1theK (원더케이)', 'Stone Music Entertainment', 'Dreamus Music',
               '워너뮤직코리아 (Warner Music Korea)', 'GENIE MUSIC']

# 각 곡별로 유통사 채널이 몇 개인지 확인
def count_dist_channels(group):
    return group['api_channel_title'].isin(REMOVE_LIST).sum()

dist_count = df_dup_deduped.groupby(['songName', 'artists']).apply(count_dist_channels)
print("유통사 채널 2개 이상인 곡:")
print(dist_count[dist_count >= 2])

유통사 채널 2개 이상인 곡:
Series([], dtype: int64)


/tmp/ipykernel_3192/250054025.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dist_count = df_dup_deduped.groupby(['songName', 'artists']).apply(count_dist_channels)


In [33]:
len(df_dup_deduped)

663

### 4-1. 유통사 채널 제거 재확인: GENIE MUSIC 추가 제거
<small>1차 제거 후에도 채널이 2개 이상 남은 곡을 다시 확인해 GENIE MUSIC을 REMOVE_LIST에 추가합니다.</small>

In [34]:
# 5차: GENIE MUSIC 제거
REMOVE_V5 = ['1theK (원더케이)', 'Stone Music Entertainment', 'Dreamus Music',
             '워너뮤직코리아 (Warner Music Korea)', 'GENIE MUSIC']

df_dup_v5 = df_dup_deduped[
    ~df_dup_deduped['api_channel_title'].isin(REMOVE_V5)
]

print(f"제거 전: {len(df_dup_deduped)} rows")
print(f"제거 후: {len(df_dup_v5)} rows")
print(f"제거된 행: {len(df_dup_deduped) - len(df_dup_v5)} rows")

result_v5 = df_dup_v5.groupby(['songName', 'artists'])['api_channel_title'].nunique()
print(f"\n아직 채널 2개 이상인 곡: {len(result_v5[result_v5 >= 2])}개")

# 남은 중복에서 어떤 채널이 많은지 확인
multi_channel_songs_v5 = result_v5[result_v5 >= 2].index
df_multi_v5 = df_dup_v5[
    df_dup_v5.set_index(['songName', 'artists']).index.isin(multi_channel_songs_v5)
]
print("\n남은 중복 채널 순위:")
print(df_multi_v5['api_channel_title'].value_counts())

제거 전: 663 rows
제거 후: 451 rows
제거된 행: 212 rows

아직 채널 2개 이상인 곡: 36개

남은 중복 채널 순위:
api_channel_title
SUPER SOUND Bugs!                           10
DanalEntertainment                           5
SEOUL MUSIC / 서울뮤직                           3
쇼파르엔터테인먼트                                    3
온리원 뮤비                                       3
TIOT 티아이오티                                   2
Brave Entertainment                          2
스튜디오:D                                       2
AIMERS                                       2
MUSIC&NEW 뮤직앤뉴                               2
Blackswan Official                           2
PARK JEUP                                    1
Official A.C.E                               1
FIFTY FIFTY Official                         1
H1GHR MUSIC                                  1
강균성 SOOM                                     1
TM ENTERTAINMENT                             1
그린유니버스뮤직 Green Universe Music (GU Music)     1
iii Official                                 1
임재현 Offi

In [36]:
len(result_v5)

410

### 4-2. 유통사 채널 제거 최종 리스트 적용
<small>남은 유통사 채널들을 추가로 확인해 REMOVE_LIST를 최종 확장한 뒤 다시 필터링합니다.</small>

In [38]:
REMOVE_LIST = ['1theK (원더케이)', 'Stone Music Entertainment', 'Dreamus Music',
             '워너뮤직코리아 (Warner Music Korea)', 'GENIE MUSIC',
             'SUPER SOUND Bugs!', 'DanalEntertainment', '소니뮤직코리아 Sony Music Korea',
             'Sound Republica', 'MUSIC&NEW 뮤직앤뉴', 'SEOUL MUSIC / 서울뮤직',
             'YOU Entertainment', 'YY Entertainment', 'AT AREA',
             'VLENDING MUSIC', '온리원 뮤비',
             'ZENITH CNM', 'TM ENTERTAINMENT', 'OGAM Entertainment',
             'Studio M-Lab', '더 볼트 THE VAULT']

# 각 곡별로 유통사 채널이 몇 개인지 확인
def count_dist_channels(group):
    return group['api_channel_title'].isin(REMOVE_LIST).sum()

dist_count = df_multi_v5.groupby(['songName', 'artists']).apply(count_dist_channels)
print("유통사 채널 2개 이상인 곡:")
print(dist_count[dist_count >= 2])

유통사 채널 2개 이상인 곡:
Series([], dtype: int64)


/tmp/ipykernel_3192/3719612530.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dist_count = df_multi_v5.groupby(['songName', 'artists']).apply(count_dist_channels)


In [39]:
df_dup_v6 = df_dup_deduped[
    ~df_dup_deduped['api_channel_title'].isin(REMOVE_LIST)
]

print(f"제거 전: {len(df_dup_deduped)} rows")
print(f"제거 후: {len(df_dup_v6)} rows")
print(f"제거된 행: {len(df_dup_deduped) - len(df_dup_v6)} rows")

result_v6 = df_dup_v6.groupby(['songName', 'artists'])['api_channel_title'].nunique()
print(f"\n아직 채널 2개 이상인 곡: {len(result_v6[result_v6 >= 2])}개")

multi_v6 = result_v6[result_v6 >= 2].index
df_dup_v6[df_dup_v6.set_index(['songName', 'artists']).index.isin(multi_v6)]\
    .groupby(['songName', 'artists'])['api_channel_title'].apply(list)

제거 전: 663 rows
제거 후: 414 rows
제거된 행: 249 rows

아직 채널 2개 이상인 곡: 2개


,,api_channel_title
songName,artists,
MOVIE,2921|박제업||0,"[그린유니버스뮤직 Green Universe Music (GU Music), PAR..."
영화 한편 본 것 같아,1561|송하예||0,"[에잇 8recordz x studio8, 플랩 [Playlist Lab]]"


In [87]:
df_multi_v5[df_multi_v5['songName']=='영화 한편 본 것 같아']

,url,songName,artists,video_id,api_view_count,api_like_count,api_comment_count,api_published_at,api_duration,api_channel_title,...,api_duration_sec,lyrics_clean,song_title_clean,lyrics_topic,song_title_topic,kor_name,activity_type,member_count,gender,is_group
389,https://www.youtube.com/watch?v=Fag8mxHphxA,영화 한편 본 것 같아,1561|송하예||0,Fag8mxHphxA,1851.0,37.0,3.0,2025-09-17 09:00:23+00:00,PT3M48S,에잇 8recordz x studio8,...,228,어느 새벽에 어김없이 넌 찾아오지 내 마음에 빈손으로 바라는 건 하나뿐인데 다시 돌...,영화 한편 본 것 같아,0,8,송하예,"여성, 솔로",1.0,여성,0.0
578,https://www.youtube.com/watch?v=i2FkyWgBMA4,영화 한편 본 것 같아,1561|송하예||0,i2FkyWgBMA4,123.0,3.0,0.0,2025-09-17 09:01:12+00:00,PT3M48S,플랩 [Playlist Lab],...,228,어느 새벽에 어김없이 넌 찾아오지 내 마음에 빈손으로 바라는 건 하나뿐인데 다시 돌...,영화 한편 본 것 같아,0,8,송하예,"여성, 솔로",1.0,여성,0.0


In [88]:
df_dup_v6[df_dup_v6['songName']=='영화 한편 본 것 같아']

,url,songName,artists,video_id,api_view_count,api_like_count,api_comment_count,api_published_at,api_duration,api_channel_title,...,api_duration_sec,lyrics_clean,song_title_clean,lyrics_topic,song_title_topic,kor_name,activity_type,member_count,gender,is_group
294,https://www.youtube.com/watch?v=Fag8mxHphxA,영화 한편 본 것 같아,1561|송하예||0,Fag8mxHphxA,1851.0,37.0,3.0,2025-09-17 09:00:23+00:00,PT3M48S,에잇 8recordz x studio8,...,228,어느 새벽에 어김없이 넌 찾아오지 내 마음에 빈손으로 바라는 건 하나뿐인데 다시 돌...,영화 한편 본 것 같아,0,8,송하예,"여성, 솔로",1.0,여성,0.0


> 같은 곡의 중복 영상이지만 유통사명 또는 video_id가 기존 항목과 겹쳐서 자동 필터(유통사 채널 제거, video_id 중복 제거)에 걸러지지 않고 남아있던 케이스라 수동으로 제거합니다.

In [54]:
df_dup_v6 = df_dup_v6.drop(
    df_dup_v6[df_dup_v6['songName'] == 'Let Me Leave You'].index
).reset_index(drop=True)

In [55]:
df_dup_v6[df_dup_v6['songName']=='Let Me Leave You']

,url,songName,artists,video_id,api_view_count,api_like_count,api_comment_count,api_published_at,api_duration,api_channel_title,...,api_duration_sec,lyrics_clean,song_title_clean,lyrics_topic,song_title_topic,kor_name,activity_type,member_count,gender,is_group


> 유통사 채널 필터 이후에도 같은 곡(`MOVIE`)이 다른 채널에 중복 업로드된 상태로 남아있어, 하나만 남기고 제거합니다.

In [58]:
df_dup_v6 = df_dup_v6.drop(
    df_dup_v6[df_dup_v6['url'] == 'https://www.youtube.com/watch?v=-Rv_K7qy4Ok'].index
).reset_index(drop=True)

In [59]:
df_dup_v6[df_dup_v6['songName']=='MOVIE']

,url,songName,artists,video_id,api_view_count,api_like_count,api_comment_count,api_published_at,api_duration,api_channel_title,...,api_duration_sec,lyrics_clean,song_title_clean,lyrics_topic,song_title_topic,kor_name,activity_type,member_count,gender,is_group
243,https://www.youtube.com/watch?v=0MWJo1C24ww,MOVIE,2921|박제업||0,0MWJo1C24ww,14493.0,1001.0,131.0,2025-07-25 09:00:02+00:00,PT3M21S,PARK JEUP,...,201,슬픈 노래는 어울리지가 않아 한 장면의 영화처럼 예쁜 기억이 나를 끌어당겨요 이제 ...,MOVIE,11,3,박제업,"남성, 솔로",1.0,남성,0.0


> 중복 확인 과정에서 발견된 저조회수 영상(`영화 한편 본 것 같아`, 조회수 1,851회)으로, 두 업로드 모두 데이터셋에서 제외합니다.

In [80]:
df_dup_v6 = df_dup_v6.drop(
    df_dup_v6[df_dup_v6['url'] == 'https://www.youtube.com/watch?v=i2FkyWgBMA4'].index
).reset_index(drop=True)

In [86]:
df_dup_v6[df_dup_v6['songName']=='영화 한편 본 것 같아']

,url,songName,artists,video_id,api_view_count,api_like_count,api_comment_count,api_published_at,api_duration,api_channel_title,...,api_duration_sec,lyrics_clean,song_title_clean,lyrics_topic,song_title_topic,kor_name,activity_type,member_count,gender,is_group
294,https://www.youtube.com/watch?v=Fag8mxHphxA,영화 한편 본 것 같아,1561|송하예||0,Fag8mxHphxA,1851.0,37.0,3.0,2025-09-17 09:00:23+00:00,PT3M48S,에잇 8recordz x studio8,...,228,어느 새벽에 어김없이 넌 찾아오지 내 마음에 빈손으로 바라는 건 하나뿐인데 다시 돌...,영화 한편 본 것 같아,0,8,송하예,"여성, 솔로",1.0,여성,0.0


In [89]:
df_dup_v6 = df_dup_v6.drop(
    df_dup_v6[df_dup_v6['url'] == 'https://www.youtube.com/watch?v=Fag8mxHphxA'].index
).reset_index(drop=True)

In [90]:
df_dup_v6[df_dup_v6['songName']=='영화 한편 본 것 같아']

,url,songName,artists,video_id,api_view_count,api_like_count,api_comment_count,api_published_at,api_duration,api_channel_title,...,api_duration_sec,lyrics_clean,song_title_clean,lyrics_topic,song_title_topic,kor_name,activity_type,member_count,gender,is_group


## 5. 중복 제거가 끝난 데이터 병합
<small>채널 정리까지 끝난 중복 후보(`df_dup_v6`)를 애초에 중복 없던 곡(`df_unique`)과 합쳐 2024년 최종본을 만듭니다.</small>

In [91]:
len(df_dup_v6)

410

In [92]:
# 1. video_id 중복 확인
print(f"video_id 중복: {df_dup_v6['video_id'].duplicated().sum()} 개")

# 2. 채널명 2개 이상인 곡 확인
result_v6 = df_dup_v6.groupby(['songName', 'artists'])['api_channel_title'].nunique()
print(f"채널명 2개 이상인 곡: {len(result_v6[result_v6 >= 2])} 개")

video_id 중복: 0 개
채널명 2개 이상인 곡: 0 개


In [93]:
len(df_unique)

1633

In [94]:
# 겹치는 곡 확인
overlap = df_unique.set_index(['songName', 'artists']).index.isin(
    df_dup_v6.set_index(['songName', 'artists']).index
)
print(f"겹치는 곡: {overlap.sum()} 개")

겹치는 곡: 0 개


In [95]:
df_final = pd.concat([df_unique, df_dup_v6]).reset_index(drop=True)
print(f"\ndf_unique: {len(df_unique)} rows")
print(f"df_dup_v6: {len(df_dup_v6)} rows")
print(f"최종: {len(df_final)} rows")


df_unique: 1633 rows
df_dup_v6: 410 rows
최종: 2043 rows


In [96]:
df_final.to_csv('data(drop_duplicated).csv', index = False)

# 2. 2023, 2025 데이터 추가
<small>2024 중복처리 결과와 비교해, 새로 크롤링한 2023,2025 데이터(`df_combined`)에서 아직 없는 곡만 추출하고 결측치 정리·채널 필터링을 거쳐 추가합니다.</small>

In [12]:
import pandas as pd
df_check = pd.read_csv('/content/df_combined(2023,2025파일) (1).csv')

In [13]:
df_check.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2828 entries, 0 to 2827
Data columns (total 57 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   url                     2828 non-null   object 
 1   songName                2828 non-null   object 
 2   artists                 2828 non-null   object 
 3   publishTime             952 non-null    object 
 4   video_id                2828 non-null   object 
 5   api_view_count          2828 non-null   float64
 6   api_like_count          2828 non-null   float64
 7   api_comment_count       2828 non-null   float64
 8   api_published_at        2827 non-null   object 
 9   api_duration            2827 non-null   object 
 10  api_title               952 non-null    object 
 11  api_channel_title       2827 non-null   object 
 12  valence_raw             2827 non-null   float64
 13  arousal_raw             2827 non-null   float64
 14  valence_normalized      2827 non-null   

In [9]:
original_df = pd.read_csv('/content/data(drop_duplicated).csv')

In [10]:
original_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2043 entries, 0 to 2042
Data columns (total 59 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   url                     2043 non-null   object 
 1   songName                2043 non-null   object 
 2   artists                 2043 non-null   object 
 3   video_id                2043 non-null   object 
 4   api_view_count          2043 non-null   float64
 5   api_like_count          2043 non-null   float64
 6   api_comment_count       2043 non-null   float64
 7   api_published_at        2043 non-null   object 
 8   api_duration            2043 non-null   object 
 9   api_channel_title       2043 non-null   object 
 10  valence_raw             2043 non-null   float64
 11  arousal_raw             2043 non-null   float64
 12  valence_normalized      2043 non-null   float64
 13  arousal_normalized      2043 non-null   float64
 14  energy                  2043 non-null   

## 1. video_id 기준 2023,2025 신규 곡 추출
<small>2024 중복처리 결과(`original_df`)와 video_id를 비교해, 아직 포함되지 않은 2023,2025 곡만 골라냅니다.</small>

In [14]:
# video_id 비교
common_ids = set(df_check['video_id']) & set(original_df['video_id'])
only_check = set(df_check['video_id']) - set(original_df['video_id'])
only_original = set(original_df['video_id']) - set(df_check['video_id'])

print(f"=== video_id 비교 ===")
print(f"df_check 총계       : {len(df_check)}")
print(f"original_df 총계    : {len(original_df)}")
print(f"video_id 같은 것    : {len(common_ids)}")
print(f"df_check에만 있는 것 : {len(only_check)}")
print(f"original_df에만 있는 것: {len(only_original)}")

# video_id 다른 것들 중 songName/artists 비교
df_only = df_check[df_check['video_id'].isin(only_check)][['video_id','songName','artists']].reset_index(drop=True)
orig_only = original_df[original_df['video_id'].isin(only_original)][['video_id','songName','artists']].reset_index(drop=True)

# songName + artists 조합으로 비교
df_only['key'] = df_only['songName'].str.strip() + '|' + df_only['artists'].str.strip()
orig_only['key'] = orig_only['songName'].str.strip() + '|' + orig_only['artists'].str.strip()

common_keys = set(df_only['key']) & set(orig_only['key'])
print(f"\n=== video_id 다른 것들 중 songName+artists 비교 ===")
print(f"songName+artists 같은 것 : {len(common_keys)}  ← video_id만 다르고 곡은 동일")
print(f"완전히 다른 곡           : {len(set(df_only['key']) - set(orig_only['key']))}")

=== video_id 비교 ===
df_check 총계       : 2828
original_df 총계    : 2043
video_id 같은 것    : 2043
df_check에만 있는 것 : 537
original_df에만 있는 것: 0

=== video_id 다른 것들 중 songName+artists 비교 ===
songName+artists 같은 것 : 0  ← video_id만 다르고 곡은 동일
완전히 다른 곡           : 502


In [15]:
df_check_only = df_check[df_check['video_id'].isin(only_check)].reset_index(drop=True)
print(f"df_check에만 있는 곡: {len(df_check_only)}개")
df_check_only

df_check에만 있는 곡: 559개


,url,songName,artists,publishTime,video_id,api_view_count,api_like_count,api_comment_count,api_published_at,api_duration,...,hapax_dislegomenon,hapax_trislegomenon,unique_tokens_per_line,avg_tokens_per_line,line_count,punctuation_ratio,compression_rate,pronoun_frequency,korean_ratio,english_ratio
0,https://www.youtube.com/watch?v=qJTEDzqD2SE,Crash,514|JAY B |JAYB|1,2024.11.27,qJTEDzqD2SE,5707077.0,27500.0,2249.0,2024-11-27 09:00:47,PT2M59S,...,38,19,4.99,5.12,80,0.0393,0.4201,0.1337,0.1831,0.5329
1,https://www.youtube.com/watch?v=506YXcMBxmc,그대만의 노래,1595|임한별||0,2024.12.09,506YXcMBxmc,1545243.0,2529.0,132.0,2024-12-09 09:00:04,PT4M56S,...,18,17,5.40,5.48,48,0.0063,0.4224,0.0879,0.8436,0.0000
2,https://www.youtube.com/watch?v=mq7PAnDk8X4,사랑할 결심,1564|오반||0,2024.12.12,mq7PAnDk8X4,786444.0,1360.0,53.0,2024-12-12 09:00:51,PT3M51S,...,21,5,6.03,6.12,58,0.0000,0.3290,0.0898,0.6432,0.2184
3,https://www.youtube.com/watch?v=7wd30pPbCnI,그녀가 웃잖아,1894|루시 (LUCY)|lucy_solo|0,2024.12.01,7wd30pPbCnI,695994.0,6731.0,328.0,2024-12-01 09:00:13,PT4M48S,...,23,9,6.08,6.26,39,0.0016,0.5275,0.0679,0.8679,0.0000
4,https://www.youtube.com/watch?v=a9QA7zOkAyQ,RESCUE TAYO,2242|Kep1er|Kep1er|1,2023.07.06,a9QA7zOkAyQ,9411927.0,29703.0,0.0,2023-07-06 09:00:01,PT2M34S,...,19,19,4.03,4.34,68,0.0088,0.3515,0.0497,0.6022,0.1906
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
554,https://www.youtube.com/watch?v=zRrq5mqjdvM,순간들,1126|로이킴|RoyKim|1,NaN,zRrq5mqjdvM,129146.0,1912.0,170.0,2025-11-30T09:01:35Z,PT3M14S,...,8,2,4.67,4.72,18,0.0000,0.6231,0.0294,0.8333,0.0000
555,https://www.youtube.com/watch?v=z_-_LOCp_IU,기도 (Ver.Acoustic),707|미연 (i-dle (아이들))|MIYEON|1,NaN,z_-_LOCp_IU,28913.0,96.0,6.0,2024-12-24T09:01:08Z,PT4M15S,...,17,31,6.84,6.87,31,0.0000,0.3519,0.1481,0.8765,0.0000
556,https://www.youtube.com/watch?v=zhI8MMjHSas,G.O.A.T,2279|H1-KEY (하이키)|H1KEY|1,NaN,zhI8MMjHSas,6884.0,323.0,24.0,2025-10-29T09:01:01Z,PT2M59S,...,11,34,5.11,5.36,75,0.0243,0.4176,0.1071,0.2962,0.4790
557,https://www.youtube.com/watch?v=zvlJEkFiWFs,AlwayS,2765|NiziU (니쥬)|NiziU|0,NaN,zvlJEkFiWFs,6029743.0,95275.0,16222.0,2024-12-02T15:01:11Z,PT4M55S,...,3,3,1.06,1.06,68,0.0280,0.4946,0.0000,0.0000,0.0000


## 2. 결측치 컬럼 정리 후 저장
<small>항상 결측인 컬럼(video_duration, video_fps, days_since_release, avg_daily_view)을 제거하고 나머지 결측 행을 정리해 `add_potential.csv`로 저장합니다.</small>

In [16]:
df_check_only.isnull().sum()

,0
url,0
songName,0
artists,0
publishTime,531
video_id,0
api_view_count,0
api_like_count,0
api_comment_count,0
api_published_at,1
api_duration,1


In [22]:
df_check_only = df_check_only.drop(['video_duration','video_fps', 'days_since_release', 'avg_daily_view'], axis = 1)

In [23]:
df_check_only.isnull().sum()

,0
url,0
songName,0
artists,0
video_id,0
api_view_count,0
api_like_count,0
api_comment_count,0
api_published_at,1
api_duration,1
api_channel_title,1


In [28]:
df_check_only.dropna(inplace=True)

In [32]:
df_check_only.isnull().sum().sum()

np.int64(0)

In [33]:
df_check_only.to_csv('add_potential.csv', index = False)

## 3. add_potential 재로드 및 후속 정리 시작
<small>저장된 `add_potential.csv`를 다시 불러와, 2024 데이터와 겹치는 곡 제거·중복 정리·채널 필터링을 이어서 진행합니다.</small>

In [75]:
add_potential = pd.read_csv('/content/add_potential.csv')

In [76]:
add_potential.isnull().sum().sum()

np.int64(0)

## 4. songName+artists 기준으로 2024 데이터와 겹치는 곡 제거
<small>video_id는 다르지만 같은 곡(songName+artists)이 2024 데이터에 이미 있는 경우를 찾아 제외합니다.</small>

In [77]:
# video_id 기준 중복 확인
common = set(original_df['video_id']) & set(add_potential['video_id'])
print(f"video_id 기준 중복: {len(common)}개")

# 혹시 video_id 달라도 같은 곡인지 확인
orig_keys = set(original_df['songName'].str.strip() + '|' + original_df['artists'].str.strip())
add_keys = set(add_potential['songName'].str.strip() + '|' + add_potential['artists'].str.strip())

common_keys = orig_keys & add_keys
print(f"songName+artists 기준 중복: {len(common_keys)}개")

video_id 기준 중복: 0개
songName+artists 기준 중복: 234개


In [78]:
# 중복 234개 상세 확인
add_potential['key'] = add_potential['songName'].str.strip() + '|' + add_potential['artists'].str.strip()
original_df['key'] = original_df['songName'].str.strip() + '|' + original_df['artists'].str.strip()

duplicated_songs = add_potential[add_potential['key'].isin(common_keys)][['video_id','songName','artists','key']]
print(f"add_potential에서 중복된 곡: {len(duplicated_songs)}개")
duplicated_songs

add_potential에서 중복된 곡: 246개


,video_id,songName,artists,key
0,qJTEDzqD2SE,Crash,514|JAY B |JAYB|1,Crash|514|JAY B |JAYB|1
1,506YXcMBxmc,그대만의 노래,1595|임한별||0,그대만의 노래|1595|임한별||0
5,gH1rVdXUlpo,에피소드,2138|이무진|LeeMujin|1,에피소드|2138|이무진|LeeMujin|1
6,ZVDksuBjUig,시나브로(Gradually),975|켄 (KEN)|KEN|1,시나브로(Gradually)|975|켄 (KEN)|KEN|1
7,K16nk_7VyL0,애상,2138|이무진|LeeMujin|1,애상|2138|이무진|LeeMujin|1
...,...,...,...,...
543,y37ye7OOLAs,Dance with me,838|정예인 (Yein)|Yein|1,Dance with me|838|정예인 (Yein)|Yein|1
544,yPOxIYxnuac,景色 (KESHIKI),2730|EVNNE(이븐)|EVNNE|1,景色 (KESHIKI)|2730|EVNNE(이븐)|EVNNE|1
547,yv_vSl0KrhY,행운을 부탁해,647|보라미유||0,행운을 부탁해|647|보라미유||0
548,ywq2ND3aK-k,그대라서 (2025 ver.),1561|송하예||0,그대라서 (2025 ver.)|1561|송하예||0


In [79]:
add_potential_clean = add_potential[~add_potential['key'].isin(common_keys)].reset_index(drop=True)
print(f"삭제 전: {len(add_potential)}개")
print(f"삭제 후: {len(add_potential_clean)}개")
print(f"삭제된 곡: {len(add_potential) - len(add_potential_clean)}개")

삭제 전: 557개
삭제 후: 311개
삭제된 곡: 246개


In [80]:
add_potential_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311 entries, 0 to 310
Data columns (total 48 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   url                     311 non-null    object 
 1   songName                311 non-null    object 
 2   artists                 311 non-null    object 
 3   video_id                311 non-null    object 
 4   api_view_count          311 non-null    float64
 5   api_like_count          311 non-null    float64
 6   api_comment_count       311 non-null    float64
 7   api_published_at        311 non-null    object 
 8   api_duration            311 non-null    object 
 9   api_channel_title       311 non-null    object 
 10  valence_raw             311 non-null    float64
 11  arousal_raw             311 non-null    float64
 12  valence_normalized      311 non-null    float64
 13  arousal_normalized      311 non-null    float64
 14  energy                  311 non-null    fl

In [81]:
# video_id 기준 중복 확인
common = set(original_df['video_id']) & set(add_potential_clean['video_id'])
print(f"video_id 기준 중복: {len(common)}개")

# 혹시 video_id 달라도 같은 곡인지 확인
orig_keys = set(original_df['songName'].str.strip() + '|' + original_df['artists'].str.strip())
add_keys = set(add_potential_clean['songName'].str.strip() + '|' + add_potential_clean['artists'].str.strip())

common_keys = orig_keys & add_keys
print(f"songName+artists 기준 중복: {len(common_keys)}개")

video_id 기준 중복: 0개
songName+artists 기준 중복: 0개


## 5. 게시일 파싱 및 video_id 중복 제거
<small>`api_published_at`을 날짜형으로 변환하고, 같은 video_id가 중복되면 조회수(`api_view_count`)가 높은 쪽을 남깁니다.</small>

In [82]:
add_potential_clean['api_published_at'] = pd.to_datetime(
    add_potential_clean['api_published_at'],
    format='mixed',
    utc=True
)
add_potential_clean['api_published_at'].dt.year.unique()

array([2024, 2023, 2025], dtype=int32)

In [71]:
add_potential_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311 entries, 0 to 310
Data columns (total 48 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   url                     311 non-null    object             
 1   songName                311 non-null    object             
 2   artists                 311 non-null    object             
 3   video_id                311 non-null    object             
 4   api_view_count          311 non-null    float64            
 5   api_like_count          311 non-null    float64            
 6   api_comment_count       311 non-null    float64            
 7   api_published_at        311 non-null    datetime64[ns, UTC]
 8   api_duration            311 non-null    object             
 9   api_channel_title       311 non-null    object             
 10  valence_raw             311 non-null    float64            
 11  arousal_raw             311 non-null    float

In [83]:
# video_id 기준 중복 확인
common = set(original_df['video_id']) & set(add_potential_clean['video_id'])
print(f"video_id 기준 중복: {len(common)}개")

# 혹시 video_id 달라도 같은 곡인지 확인
orig_keys = set(original_df['songName'].str.strip() + '|' + original_df['artists'].str.strip())
add_keys = set(add_potential_clean['songName'].str.strip() + '|' + add_potential_clean['artists'].str.strip())

common_keys = orig_keys & add_keys
print(f"songName+artists 기준 중복: {len(common_keys)}개")

video_id 기준 중복: 0개
songName+artists 기준 중복: 0개


In [84]:
print(add_potential_clean['video_id'].nunique())
print(len(add_potential_clean))

299
311


In [86]:
add_potential_clean.columns

Index(['url', 'songName', 'artists', 'video_id', 'api_view_count',
       'api_like_count', 'api_comment_count', 'api_published_at',
       'api_duration', 'api_channel_title', 'valence_raw', 'arousal_raw',
       'valence_normalized', 'arousal_normalized', 'energy', 'loudness',
       'tempo', 'duration_ms', 'speechiness', 'acoustic_score',
       'sentiment_compound', 'sentiment_positive', 'sentiment_negative',
       'sentiment_neutral', 'avg_brightness', 'avg_motion', 'avg_r_value',
       'avg_g_value', 'avg_b_value', 'frames_analyzed', 'genie_lyrics',
       'genie_genre', 'token_count', 'unique_token_ratio',
       'repeated_token_ratio', 'avg_token_length', 'hapax_legomenon',
       'hapax_dislegomenon', 'hapax_trislegomenon', 'unique_tokens_per_line',
       'avg_tokens_per_line', 'line_count', 'punctuation_ratio',
       'compression_rate', 'pronoun_frequency', 'korean_ratio',
       'english_ratio', 'key'],
      dtype='object')

In [90]:
dup = add_potential_clean[add_potential_clean['video_id'].duplicated(keep=False)]
print(f"중복 video_id 행: {len(dup)}개")
dup.sort_values('video_id')[['video_id', 'songName', 'artists','api_channel_title','api_view_count','url']]

중복 video_id 행: 24개


,video_id,songName,artists,api_channel_title,api_view_count,url
9,6_YAKarZq48,BURN IT,290|FTISLAND (FT아일랜드)|FTISLAND|1,1theK (원더케이),98145.0,https://www.youtube.com/watch?v=6_YAKarZq48
48,6_YAKarZq48,BURN IT,290|FTISLAND (FT아일랜드)|FTISLAND|1,1theK (원더케이),98128.0,https://www.youtube.com/watch?v=6_YAKarZq48
1,7wd30pPbCnI,그녀가 웃잖아,1894|루시 (LUCY)|lucy_solo|0,GENIE MUSIC,695994.0,https://www.youtube.com/watch?v=7wd30pPbCnI
55,7wd30pPbCnI,그녀가 웃잖아,1894|루시 (LUCY)|lucy_solo|0,GENIE MUSIC,695368.0,https://www.youtube.com/watch?v=7wd30pPbCnI
12,F1NOCLS7VQQ,T.I.E (Take It Easy) (feat. 박재범),1218|범키|BUMKEY|1,1theK (원더케이),21974.0,https://www.youtube.com/watch?v=F1NOCLS7VQQ
98,F1NOCLS7VQQ,T.I.E (Take It Easy) (feat. 박재범),1218|범키|BUMKEY|1,1theK (원더케이),21956.0,https://www.youtube.com/watch?v=F1NOCLS7VQQ
119,H_QdYFMiEf0,Dear,312|러블리즈|Lovelyz|1,1theK (원더케이),45777.0,https://www.youtube.com/watch?v=H_QdYFMiEf0
7,H_QdYFMiEf0,Dear,312|러블리즈|Lovelyz|1,1theK (원더케이),45806.0,https://www.youtube.com/watch?v=H_QdYFMiEf0
3,HjGQSp3sHnw,PINK CLOUD,455|츄(Chuu)|Chuu|1,1theK (원더케이),334304.0,https://www.youtube.com/watch?v=HjGQSp3sHnw
121,HjGQSp3sHnw,PINK CLOUD,455|츄(Chuu)|Chuu|1,1theK (원더케이),334226.0,https://www.youtube.com/watch?v=HjGQSp3sHnw


In [91]:
add_potential_clean = (
    add_potential_clean
    .sort_values('api_view_count', ascending=False)
    .drop_duplicates(subset='video_id', keep='first')
    .reset_index(drop=True)
)

print(f"중복 제거 후: {len(add_potential_clean)}개")
print(f"video_id 고유값: {add_potential_clean['video_id'].nunique()}개")

중복 제거 후: 299개
video_id 고유값: 299개


In [92]:
add_potential_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 48 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   url                     299 non-null    object             
 1   songName                299 non-null    object             
 2   artists                 299 non-null    object             
 3   video_id                299 non-null    object             
 4   api_view_count          299 non-null    float64            
 5   api_like_count          299 non-null    float64            
 6   api_comment_count       299 non-null    float64            
 7   api_published_at        299 non-null    datetime64[ns, UTC]
 8   api_duration            299 non-null    object             
 9   api_channel_title       299 non-null    object             
 10  valence_raw             299 non-null    float64            
 11  arousal_raw             299 non-null    float

## 6. songName+artists 기준 잔여 중복 확인
<small>video_id 중복 제거 후에도 같은 곡이 남아있는지 곡 단위로 다시 확인합니다.</small>

In [96]:
print(add_potential_clean['songName'].nunique())
print(add_potential_clean['artists'].nunique())

265
210


In [98]:
add_potential_clean['key'] = add_potential_clean['songName'].str.strip() + '|' + add_potential_clean['artists'].str.strip()

print(f"전체 행수       : {len(add_potential_clean)}")
print(f"video_id 고유값 : {add_potential_clean['video_id'].nunique()}")
print(f"songName+artists 고유값: {add_potential_clean['key'].nunique()}")

# 중복된 곡 확인
dup_keys = add_potential_clean[add_potential_clean['key'].duplicated(keep=False)]
print(f"\nsongName+artists 기준 중복: {len(dup_keys)}개")
dup_keys.sort_values('key')[['video_id', 'songName', 'artists', 'api_channel_title','api_view_count']]

전체 행수       : 299
video_id 고유값 : 299
songName+artists 고유값: 266

songName+artists 기준 중복: 65개


,video_id,songName,artists,api_channel_title,api_view_count
6,sswmsnPigDc,Can't Stop,653|투모로우바이투게더|TXT|1,HYBE LABELS,25195788.0
279,GtYCop5hsnI,Can't Stop,653|투모로우바이투게더|TXT|1,뮤직버디 MusicBuddy,5777.0
75,e0eutjs-Aj4,Can't Stop,653|투모로우바이투게더|TXT|1,GENIE MUSIC,822965.0
297,KW9n_UGBNjU,Clumsy,295|넬 (NELL)|NELL|1,뮤직버디 MusicBuddy,784.0
227,GkAEJKSAhrs,Clumsy,295|넬 (NELL)|NELL|1,GENIE MUSIC,35961.0
...,...,...,...,...,...
284,QRF7sxZm7gg,이음선(TIMELORD) (Narr. 온달),601|용훈 (ONEWE) |Yong Hoon|0,MUSIC&NEW 뮤직앤뉴,4456.0
202,4dBNQIg5bic,잃어버린 별,1129|CHEEZE (치즈)|CHEEZE|1,GENIE MUSIC,64432.0
244,uMCohsIevU0,잃어버린 별,1129|CHEEZE (치즈)|CHEEZE|1,모스트콘텐츠 MOSTCONTENTS,21132.0
155,4bbhHNqL9OE,헹가래,1123|개코||0,1theK (원더케이),148058.0


## 7. 채널별 분포 확인 및 공식 채널만 필터링
<small>채널명 분포를 확인하고, 공식 아티스트 채널(`official_channels`) 목록에 해당하는 영상만 남깁니다.</small>

In [99]:
print(add_potential_clean['api_channel_title'].value_counts())

api_channel_title
1theK (원더케이)                 48
GENIE MUSIC                  31
HYBE LABELS                  21
Stone Music Entertainment    15
SUPER SOUND Bugs!            14
                             ..
바비킴 (Bobby Kim)               1
E-MOTION STUDIO               1
도넛가게                          1
에잇 8recordz x studio8         1
플랩 [Playlist Lab]             1
Name: count, Length: 84, dtype: int64


In [100]:
print(add_potential_clean['api_channel_title'].unique().tolist())

['League of Legends', 'AKMU', 'HYBE LABELS', 'Stray Kids Japan Official YouTube', 'TWICE JAPAN OFFICIAL YouTube Channel', 'SMTOWN', 'JYP Entertainment', 'ITZY JAPAN OFFICIAL YouTube Channel', 'SEOUL MUSIC / 서울뮤직', 'Sony Music (Japan)', '뽀로로와 노래해요 (뽀로로 • 타요 인기동요)', 'DAESUNG', 'BingCrosbyVEVO', 'THEBLACKLABEL', 'NiziU Official', '영탁스클럽 YOUNGTAKsClub', 'STUDIO LICO', 'KQ ENTERTAINMENT', '1theK (원더케이)', 'Billlie', '하현상 HA HYUN SANG', 'tripleS', '장민호', 'GENIE MUSIC', '이찬원', '송가인', 'Stone Music Entertainment', 'Victor Entertainment', 'MUSIC&NEW 뮤직앤뉴', 'STAYC Japan Official', '리베란테 Libelante', '스튜디오:D', '모스트콘텐츠 MOSTCONTENTS', '워너뮤직코리아 (Warner Music Korea)', 'SUPER SOUND Bugs!', '라포엠 LA POEM', 'MAMAMOO', '피버스 Feverse', '스튜디오 마음C ', '日本コロムビア 公式YouTubeチャンネル', 'VLENDING 블렌딩', 'Netflix Korea 넷플릭스 코리아', '김명수 KIM MYUNGSOO L', 'MLD ENTERTAINMENT', 'New Era Project (NEP)', 'SBS Catch', 'BIGOCEAN ENM', 'TOON STUDIO', '2PM Japan Official YouTube Channel', 'Dreamus Music', '쇼박스 SHOWBOX', '플레이다 Playda', '

In [105]:
add_potential_clean[add_potential_clean['api_channel_title'] == '제나두엔터테인먼트 XANADU_ENT_OFFICIAL ']

,url,songName,artists,video_id,api_view_count,api_like_count,api_comment_count,api_published_at,api_duration,api_channel_title,...,hapax_trislegomenon,unique_tokens_per_line,avg_tokens_per_line,line_count,punctuation_ratio,compression_rate,pronoun_frequency,korean_ratio,english_ratio,key
277,https://www.youtube.com/watch?v=Bvl4IyRL2ho,그때 우리로 돌아갈 수 있을까요,692|권은비|KWON_EUN_BI|1,Bvl4IyRL2ho,6513.0,211.0,18.0,2023-12-13 09:00:11+00:00,PT4M9S,제나두엔터테인먼트 XANADU_ENT_OFFICIAL,...,10,6.43,6.5,28,0.0,0.4846,0.0769,0.875,0.0,그때 우리로 돌아갈 수 있을까요|692|권은비|KWON_EUN_BI|1


In [107]:
add_potential_clean[add_potential_clean['api_channel_title'] == '제나두엔터테인먼트 XANADU_ENT_OFFICIAL ']['genie_lyrics']

,genie_lyrics
277,바람이 나를 스치며 너의 흔적을 가져가요\n잡아도 흩어져 가는 너를 그저 바라보다\...


In [108]:
add_potential_clean[add_potential_clean['api_channel_title'] == 'E-MOTION STUDIO']

,url,songName,artists,video_id,api_view_count,api_like_count,api_comment_count,api_published_at,api_duration,api_channel_title,...,hapax_trislegomenon,unique_tokens_per_line,avg_tokens_per_line,line_count,punctuation_ratio,compression_rate,pronoun_frequency,korean_ratio,english_ratio,key
293,https://www.youtube.com/watch?v=LQKvE82r06Y,구름꽃,2531|경서||0,LQKvE82r06Y,2323.0,91.0,12.0,2025-04-15 09:00:26+00:00,PT3M54S,E-MOTION STUDIO,...,23,7.83,7.83,24,0.0,0.3564,0.1422,0.891,0.0,구름꽃|2531|경서||0


In [122]:
official_channels = [
    'HYBE LABELS', 'SMTOWN', 'JYP Entertainment', 'THEBLACKLABEL',
    'H1GHR MUSIC', 'MAMAMOO', 'AKMU', '제나두엔터테인먼트 XANADU_ENT_OFFICIAL ',
    '바비킴 (Bobby Kim)'

]

In [126]:
add_potential_clean[add_potential_clean['api_channel_title'] == 'HYBE LABELS']

,url,songName,artists,video_id,api_view_count,api_like_count,api_comment_count,api_published_at,api_duration,api_channel_title,...,hapax_trislegomenon,unique_tokens_per_line,avg_tokens_per_line,line_count,punctuation_ratio,compression_rate,pronoun_frequency,korean_ratio,english_ratio,key
2,https://www.youtube.com/watch?v=KqE0P1qMtQg,Back to Life,2540|&TEAM|&TEAM|1,KqE0P1qMtQg,38836671.0,200293.0,55127.0,2025-10-27 08:58:07+00:00,PT4M27S,HYBE LABELS,...,39,5.01,5.12,76,0.0029,0.3399,0.1034,0.3858,0.4440,Back to Life|2540|&TEAM|&TEAM|1
5,https://www.youtube.com/watch?v=HAWYOuMGkK0,Winter Ahead (with 박효신),"409|V|V|1,1080|박효신|ParkHyoShin|1",HAWYOuMGkK0,25540023.0,1689067.0,165555.0,2024-11-29 05:00:00+00:00,PT6M23S,HYBE LABELS,...,11,4.98,5.05,43,0.0185,0.4231,0.1051,0.0000,0.7782,"Winter Ahead (with 박효신)|409|V|V|1,1080|박효신|Par..."
6,https://www.youtube.com/watch?v=sswmsnPigDc,Can't Stop,653|투모로우바이투게더|TXT|1,sswmsnPigDc,25195788.0,286524.0,19487.0,2025-10-19 14:58:06+00:00,PT2M39S,HYBE LABELS,...,3,4.14,4.22,37,0.0337,0.3987,0.1042,0.0000,0.5729,Can't Stop|653|투모로우바이투게더|TXT|1
9,https://www.youtube.com/watch?v=HFZUAXhdnHk,DIFFERENT,2380|LE SSERAFIM (르세라핌)|LESSERAFIM|1,HFZUAXhdnHk,22245415.0,379055.0,11225.0,2025-06-08 15:00:05+00:00,PT2M23S,HYBE LABELS,...,4,3.66,3.88,41,0.0151,0.4900,0.0924,0.0000,0.6576,DIFFERENT|2380|LE SSERAFIM (르세라핌)|LESSERAFIM|1
10,https://www.youtube.com/watch?v=HeqsjDF7Lw0,Toki Yo Tomare,2853|아일릿(ILLIT)|ILLIT|1,HeqsjDF7Lw0,18416849.0,189284.0,9260.0,2025-08-31 14:58:06+00:00,PT3M30S,HYBE LABELS,...,5,1.69,1.69,78,0.0416,0.3824,0.0000,0.0000,0.1914,Toki Yo Tomare|2853|아일릿(ILLIT)|ILLIT|1
12,https://www.youtube.com/watch?v=WklwroX2yOU,Aoarashi,2540|&TEAM|&TEAM|1,WklwroX2yOU,15849119.0,142420.0,40507.0,2024-08-06 15:00:02+00:00,PT3M24S,HYBE LABELS,...,12,4.98,5.02,51,0.0014,0.4845,0.1212,0.8902,0.0758,Aoarashi|2540|&TEAM|&TEAM|1
13,https://youtu.be/RjJTiajpyrA,Go in Blind,2540|&TEAM|&TEAM|1,RjJTiajpyrA,14020728.0,149094.0,31865.0,2025-04-21 11:00:03+00:00,PT3M50S,HYBE LABELS,...,12,5.96,6.28,57,0.0257,0.4983,0.0706,0.3698,0.4282,Go in Blind|2540|&TEAM|&TEAM|1
15,https://www.youtube.com/watch?v=0V6zjDb1YN0,Yukiakari,2540|&TEAM|&TEAM|1,0V6zjDb1YN0,12667280.0,100228.0,50239.0,2024-12-15 15:00:01+00:00,PT4M51S,HYBE LABELS,...,11,5.95,5.95,37,0.0000,0.4637,0.0437,0.9607,0.0000,Yukiakari|2540|&TEAM|&TEAM|1
16,https://www.youtube.com/watch?v=ogmUm0xh8-w,0%,3708|SANTOS BRAVOS|SANTOS_BRAVOS|0,ogmUm0xh8-w,11921977.0,119153.0,15354.0,2025-12-19 00:00:00+00:00,PT3M21S,HYBE LABELS,...,0,5.00,5.00,1,0.0833,1.3667,0.0000,0.8000,0.0000,0%|3708|SANTOS BRAVOS|SANTOS_BRAVOS|0
17,https://www.youtube.com/watch?v=DY4ckIx94xw,Kawaii (Prod. Gen Hoshino),2380|LE SSERAFIM (르세라핌)|LESSERAFIM|1,DY4ckIx94xw,10947168.0,245377.0,5936.0,2025-07-08 14:58:06+00:00,PT3M49S,HYBE LABELS,...,8,5.12,5.44,89,0.0273,0.3126,0.1381,0.0000,0.7522,Kawaii (Prod. Gen Hoshino)|2380|LE SSERAFIM (르...


In [123]:
official_only = add_potential_clean[add_potential_clean['api_channel_title'].isin(official_channels)]
print(f"공식 채널 해당 곡: {len(official_only)}개")
print(official_only['api_channel_title'].value_counts())

공식 채널 해당 곡: 44개
api_channel_title
HYBE LABELS                       21
SMTOWN                            13
JYP Entertainment                  4
AKMU                               1
THEBLACKLABEL                      1
MAMAMOO                            1
H1GHR MUSIC                        1
제나두엔터테인먼트 XANADU_ENT_OFFICIAL      1
바비킴 (Bobby Kim)                    1
Name: count, dtype: int64


In [124]:
official_only.info()

<class 'pandas.core.frame.DataFrame'>
Index: 44 entries, 1 to 289
Data columns (total 48 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   url                     44 non-null     object             
 1   songName                44 non-null     object             
 2   artists                 44 non-null     object             
 3   video_id                44 non-null     object             
 4   api_view_count          44 non-null     float64            
 5   api_like_count          44 non-null     float64            
 6   api_comment_count       44 non-null     float64            
 7   api_published_at        44 non-null     datetime64[ns, UTC]
 8   api_duration            44 non-null     object             
 9   api_channel_title       44 non-null     object             
 10  valence_raw             44 non-null     float64            
 11  arousal_raw             44 non-null     float64    

## 8. 2024 데이터와 최종 비교
<small>최종 필터링된 2023,2025 추가분(`official_only`)을 2024 결과(`original_df`)와 비교해 이상치를 확인합니다.</small>

In [125]:
original_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2043 entries, 0 to 2042
Data columns (total 60 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   url                     2043 non-null   object 
 1   songName                2043 non-null   object 
 2   artists                 2043 non-null   object 
 3   video_id                2043 non-null   object 
 4   api_view_count          2043 non-null   float64
 5   api_like_count          2043 non-null   float64
 6   api_comment_count       2043 non-null   float64
 7   api_published_at        2043 non-null   object 
 8   api_duration            2043 non-null   object 
 9   api_channel_title       2043 non-null   object 
 10  valence_raw             2043 non-null   float64
 11  arousal_raw             2043 non-null   float64
 12  valence_normalized      2043 non-null   float64
 13  arousal_normalized      2043 non-null   float64
 14  energy                  2043 non-null   

In [129]:
original_df[original_df['artists'].str.contains('LE SSERAFIM ')]

,url,songName,artists,video_id,api_view_count,api_like_count,api_comment_count,api_published_at,api_duration,api_channel_title,...,lyrics_clean,song_title_clean,lyrics_topic,song_title_topic,kor_name,activity_type,member_count,gender,is_group,key
24,https://www.youtube.com/watch?v=bNKXxwOQYB8,EASY,2380|LE SSERAFIM (르세라핌)|LESSERAFIM|1,bNKXxwOQYB8,116968954.0,1470085.0,58794.0,2024-02-19 08:58:08+00:00,PT3M10S,HYBE LABELS,...,다친대도 길을 걸어 kiss me 쉽지 않음 내가 쉽게 easy Stage 위엔 불...,EASY,12,4,LE SSERAFIM (르세라핌),"여성, 그룹",5.0,여성,1.0,EASY|2380|LE SSERAFIM (르세라핌)|LESSERAFIM|1
478,https://www.youtube.com/watch?v=cB4c6ZjBBZw,EASY (English ver.),2380|LE SSERAFIM (르세라핌)|LESSERAFIM|1,cB4c6ZjBBZw,866995.0,54624.0,981.0,2024-02-23 04:58:09+00:00,PT2M46S,HYBE LABELS,...,Copy cause they mad that I still kiss me Shit ...,EASY English ver,9,3,LE SSERAFIM (르세라핌),"여성, 그룹",5.0,여성,1.0,EASY (English ver.)|2380|LE SSERAFIM (르세라핌)|LE...
1192,https://www.youtube.com/watch?v=Srn1SVbp9KE,"이브, 프시케 그리고 푸른 수염의 아내 (feat. UPSAHL)",2380|LE SSERAFIM (르세라핌)|LESSERAFIM|1,Srn1SVbp9KE,1019829.0,63056.0,1190.0,2023-07-14 04:00:00+00:00,PT3M7S,HYBE LABELS,...,I m a mess mess mess mess mess mess mess I m a...,이브 프시케 그리고 푸른 수염의 아내 feat UPSAHL,1,5,LE SSERAFIM (르세라핌),"여성, 그룹",5.0,여성,1.0,"이브, 프시케 그리고 푸른 수염의 아내 (feat. UPSAHL)|2380|LE S..."
1209,https://www.youtube.com/watch?v=TvVtYaqCni8,SPAGHETTI (feat. j-hope of BTS),2380|LE SSERAFIM (르세라핌)|LESSERAFIM|1,TvVtYaqCni8,86017927.0,1663102.0,76048.0,2025-10-24 03:58:08+00:00,PT3M19S,HYBE LABELS,...,This is a hot spot 숨 쉬듯 찾는 네 밥상 단골이 된 넌 fall i...,SPAGHETTI feat j hope of BTS,1,5,LE SSERAFIM (르세라핌),"여성, 그룹",5.0,여성,1.0,SPAGHETTI (feat. j-hope of BTS)|2380|LE SSERAF...
1220,https://www.youtube.com/watch?v=UBURTj20HXI,UNFORGIVEN (feat. Nile Rodgers),2380|LE SSERAFIM (르세라핌)|LESSERAFIM|1,UBURTj20HXI,144797274.0,2058316.0,70889.0,2023-05-01 08:58:10+00:00,PT4M20S,HYBE LABELS,...,Unforgiven I m a villain I m a Unforgiven 난 그 ...,UNFORGIVEN feat Nile Rodgers,3,5,LE SSERAFIM (르세라핌),"여성, 그룹",5.0,여성,1.0,UNFORGIVEN (feat. Nile Rodgers)|2380|LE SSERAF...
1230,https://www.youtube.com/watch?v=Uf4nGafC-zw,Perfect Night (Holiday Remix),2380|LE SSERAFIM (르세라핌)|LESSERAFIM|1,Uf4nGafC-zw,1851454.0,65522.0,1361.0,2023-11-23 04:58:09+00:00,PT2M43S,HYBE LABELS,...,Me and my girlies We gon party til its early G...,Perfect Night Holiday Remix,9,3,LE SSERAFIM (르세라핌),"여성, 그룹",5.0,여성,1.0,Perfect Night (Holiday Remix)|2380|LE SSERAFIM...
1356,https://www.youtube.com/watch?v=dZs_cLHfnNA,"이브, 프시케 그리고 푸른 수염의 아내",2380|LE SSERAFIM (르세라핌)|LESSERAFIM|1,dZs_cLHfnNA,131834516.0,1611834.0,25566.0,2023-05-23 14:58:08+00:00,PT3M48S,HYBE LABELS,...,I m a mess mess mess mess mess mess mess I m a...,이브 프시케 그리고 푸른 수염의 아내,12,5,LE SSERAFIM (르세라핌),"여성, 그룹",5.0,여성,1.0,"이브, 프시케 그리고 푸른 수염의 아내|2380|LE SSERAFIM (르세라핌)|..."
1409,https://www.youtube.com/watch?v=hLvWy2b857I,Perfect Night,2380|LE SSERAFIM (르세라핌)|LESSERAFIM|1,hLvWy2b857I,137418464.0,1315758.0,30012.0,2023-10-27 03:58:10+00:00,PT3M3S,HYBE LABELS,...,Me and my girlies We gon party til its early G...,Perfect Night,9,1,LE SSERAFIM (르세라핌),"여성, 그룹",5.0,여성,1.0,Perfect Night|2380|LE SSERAFIM (르세라핌)|LESSERAF...
1541,https://www.youtube.com/watch?v=r9AEGPB6qIU,HOT,2380|LE SSERAFIM (르세라핌)|LESSERAFIM|1,r9AEGPB6qIU,73496576.0,651744.0,30838.0,2025-03-14 03:58:07+00:00,PT2M49S,HYBE LABELS,...,위태로운 드라이브 바꿔 넣어 gear 불타는 노을 너와 내 tears so Don ...,HOT,13,1,LE SSERAFIM (르세라핌),"여성, 그룹",5.0,여성,1.0,HOT|2380|LE SSERAFIM (르세라핌)|LESSERAFIM|1
1645,https://www.youtube.com/watch?v=n6B5gQXlB-0,CRAZY,2380|LE SSERAFIM (르세라핌)|LESSERAFIM|1,n6B5gQXlB-0,195858257.0,1768991.0,72033.0,2024-08-30 03:58:08+00:00,PT2M50S,HYBE LABELS,...,Act like an angel and dress like crazy All the...,CRAZY,8,4,LE SSERAFIM (르세라핌),"여성, 그룹",5.0,여성,1.0,CRAZY|2380|LE SSERAFIM (르세라핌)|LESSERAFIM|1


In [131]:
print(len(original_df))
print(original_df['video_id'].nunique())

2043
2043


# 3. 최종 병합 및 저장
<small>2024 중복처리 결과(`original_df`)와 2023,2025 추가분(`official_only`)을 합쳐 최종 데이터를 저장합니다.</small>

In [ ]:
# 2024 중복처리 결과(original_df) + 2023,2025 신규 곡(official_only) 최종 병합
# official_only는 add_potential_clean에서 파생되며 임시 작업 컬럼('key')을 가지고 있어 제거 후 합칩니다.
final_combined_df = pd.concat(
    [original_df, official_only.drop(columns=['key'], errors='ignore')],
    ignore_index=True
)

print(f"2024 중복처리 결과: {len(original_df)} rows")
print(f"2023,2025 신규 추가: {len(official_only)} rows")
print(f"최종 합계: {len(final_combined_df)} rows")

final_combined_df.to_csv('전체뮤비_최종전처리완료.csv', index=False)